In [ ]:
import pandas as pd
import numpy as np

df_train = pd.read_csv('../data/delay_predictor_dataset.csv')
sample_trips = df_train[df_train['trip_id'] == '5.002'].copy()

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    # Radius bumi dalam kilometer
    R = 6371.0
    
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)
    
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    a = np.sin(dlat / 2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    
    return R * c

In [ ]:
# Kelompokkan per rute segmen di dalam trip untuk menghitung proxy linear
sample_trips['dtd_proxy'] = np.nan

for seg, group in sample_trips.groupby('location'):
    if len(group) > 1:
        # Titik akhir segmen diasumsikan ada pada koordinat baris terakhir di segmen tersebut
        end_lat = group['latitude'].iloc[-1]
        end_lon = group['longitude'].iloc[-1]
        
        # Titik awal segmen ada pada baris pertama segmen tersebut
        start_lat = group['latitude'].iloc[0]
        start_lon = group['longitude'].iloc[0]
        
        # Hitung jarak total segmen
        seg_len = haversine_distance(start_lat, start_lon, end_lat, end_lon)
        
        if seg_len > 0:
            # Hitung jarak real-time ke ujung segmen
            dist_to_end = haversine_distance(group['latitude'], group['longitude'], end_lat, end_lon)
            
            # Hitung dtd_proxy sesuai rumus bimbingan
            proxy_vals = 1 - (dist_to_end / seg_len)
            sample_trips.loc[group.index, 'dtd_proxy'] = np.clip(proxy_vals, 0, 1)

# Buang baris yang tidak memiliki nilai proxy (jika ada data tunggal/NaN)
sample_trips_clean = sample_trips.dropna(subset=['dtd_proxy', 'dtd'])

In [ ]:
from sklearn.metrics import r2_score
from scipy.stats import pearsonr

r2 = r2_score(sample_trips_clean['dtd'], sample_trips_clean['dtd_proxy'])
corr, _ = pearsonr(sample_trips_clean['dtd'], sample_trips_clean['dtd_proxy'])

print(f"Nilai R² (Koefisien Determinasi): {r2:.4f}")
print(f"Koefisien Korelasi Pearson: {corr:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from scipy.stats import pearsonr

df = pd.read_csv('../data/delay_predictor_dataset.csv')

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0  # Radius bumi dalam kilometer
    lat1_rad, lon1_rad = np.radians(lat1), np.radians(lon1)
    lat2_rad, lon2_rad = np.radians(lat2), np.radians(lon2)
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    a = np.sin(dlat / 2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

unique_trips = df['trip_id'].unique()
np.random.seed(42)  
sample_trip_ids = np.random.choice(unique_trips, size=30, replace=False)

df_large_sample = df[df['trip_id'].isin(sample_trip_ids)].copy()

df_large_sample['dtd_proxy'] = np.nan

for (trip, seg), group in df_large_sample.groupby(['trip_id', 'location']):
    if len(group) > 1:
        start_lat = group['latitude'].iloc[0]
        start_lon = group['longitude'].iloc[0]
        
        end_lat = group['latitude'].iloc[-1]
        end_lon = group['longitude'].iloc[-1]
        
        seg_len = haversine_distance(start_lat, start_lon, end_lat, end_lon)
        
        if seg_len > 0:
            dist_to_end = haversine_distance(group['latitude'], group['longitude'], end_lat, end_lon)
            
            # Rumus proxy: 1 - (sisa_jarak / total_jarak_segmen)
            proxy_vals = 1 - (dist_to_end / seg_len)
            
            # Clip nilai ke rentang [0, 1] 
            df_large_sample.loc[group.index, 'dtd_proxy'] = np.clip(proxy_vals, 0, 1)

df_test_final = df_large_sample.dropna(subset=['dtd_proxy', 'dtd'])

r2_global = r2_score(df_test_final['dtd'], df_test_final['dtd_proxy'])
pearson_global, _ = pearsonr(df_test_final['dtd'], df_test_final['dtd_proxy'])

print(f"=== HASIL VALIDASI EMPIRIS SKALA BESAR ===")
print(f"Jumlah Trip yang Diuji : {len(sample_trip_ids)} trip")
print(f"Total Baris Data diuji : {len(df_test_final)} baris")
print(f"Nilai R² (Global)      : {r2_global:.4f}")
print(f"Korelasi Pearson       : {pearson_global:.4f}")

In [ ]:
df = pd.read_csv('../data/delay_predictor_dataset.csv')
df.head()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

df_raw = pd.read_csv('../data/delay_predictor_dataset.csv', low_memory=False)
print(f"Data awal berhasil dimuat: {df_raw.shape} baris, {df_raw.shape[1]} kolom")

df_cleaned = df_raw.drop_duplicates().copy()
print(f"Ukuran setelah drop duplicates: {df_cleaned.shape}")

df_cleaned = df_cleaned[df_cleaned['corridor'].astype(str) != '0'].copy()
print(f"Ukuran setelah membuang anomali koridor '0': {df_cleaned.shape}")


datetime_col = 'gps_datetime' if 'gps_datetime' in df_cleaned.columns else 'time'

if datetime_col in df_cleaned.columns:
    df_cleaned[datetime_col] = pd.to_datetime(df_cleaned[datetime_col])
    df_cleaned['hour'] = df_cleaned[datetime_col].dt.hour
    df_cleaned['day_of_week'] = df_cleaned[datetime_col].dt.dayofweek
    print("Fitur 'hour' dan 'day_of_week' berhasil diekstrak.")
else:
    print("[Peringatan] Kolom datetime tidak ditemukan, pastikan fitur 'hour' dan 'day_of_week' sudah ada.")


df_cleaned['speed_clipped'] = df_cleaned['speed'].clip(lower=1.5)

# Kalkulasi ETA Estorasi (menggunakan baseline speed 50 km/jam) dan dikonversi ke menit (* 60)
df_cleaned['estimated_eta'] = df_cleaned['dtd'] / 50 * 60

# Kalkulasi ETA Aktual berdasarkan kecepatan real-time yang sudah di-clip
df_cleaned['actual_eta'] = df_cleaned['dtd'] / df_cleaned['speed_clipped'] * 60

# Selisih antara ETA Aktual dan ETA Estimasi menjadi target Regresi: delay_minutes
df_cleaned['delay_minutes'] = df_cleaned['actual_eta'] - df_cleaned['estimated_eta']
print("Kalkulasi fitur target 'delay_minutes' selesai.")


le = LabelEncoder()
df_cleaned['corridor'] = le.fit_transform(df_cleaned['corridor'].astype(str))
print("Label Encoding untuk fitur 'corridor' selesai.")


df_model = df_cleaned.copy()
print(f"Feature engineering selesai! DataFrame 'df_model' siap digunakan.")
print(f"Ukuran Akhir Data: {df_model.shape}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

df = pd.read_csv('../data/delay_predictor_dataset.csv', low_memory=False)
print(f"Data berhasil dimuat. Ukuran awal: {df.shape}")

df = df[df['corridor'].astype(str) != '0'].copy()
print(f"Ukuran data setelah filter koridor tidak valid: {df.shape}")

df['speed_clipped'] = df['speed'].clip(lower=1.5)
df['estimated_eta'] = df['dtd'] / 50 * 60
df['actual_eta'] = df['dtd'] / df['speed_clipped'] * 60
df['delay_minutes'] = df['actual_eta'] - df['estimated_eta']
print("Kalkulasi target 'delay_minutes' selesai.")

le = LabelEncoder()
df['corridor'] = le.fit_transform(df['corridor'].astype(str))
print("Label Encoding untuk kolom 'corridor' selesai.")

features_baru = ['hour', 'day_of_week', 'corridor']
target_col = 'delay_minutes'

X = df[features_baru]
y_reg = df[target_col]

X_train, X_test, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)

print(f"Jumlah data training : {X_train.shape[0]} baris")
print(f"Fitur yang digunakan  : {list(X_train.columns)}")
print(f"Target Regresi        : {target_col}")

rf_reg_baru = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg_baru.fit(X_train, y_train_reg)
print("Training selesai!")

y_pred_reg = rf_reg_baru.predict(X_test)
r2_baru = r2_score(y_test_reg, y_pred_reg)
mae_baru = mean_absolute_error(y_test_reg, y_pred_reg)
rmse_baru = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))

print(f"R² Score (Baru) : {r2_baru:.4f}")
print(f"MAE (Baru)      : {mae_baru:.2f} menit")
print(f"RMSE (Baru)     : {rmse_baru:.2f} menit")

importances = rf_reg_baru.feature_importances_
for feat, imp in zip(features_baru, importances):
    print(f"{feat}: {imp:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

df = pd.read_csv('../data/delay_predictor_dataset.csv', low_memory=False)
df = df[df['corridor'].astype(str) != '0'].copy()
print(f"Ukuran data: {df.shape}")

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1_rad, lon1_rad = np.radians(lat1), np.radians(lon1)
    lat2_rad, lon2_rad = np.radians(lat2), np.radians(lon2)
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    a = np.sin(dlat / 2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

df['distance_to_stop'] = np.nan

for (trip, seg), group in df.groupby(['trip_id', 'location']):
    if len(group) > 1:
        end_lat = group['latitude'].iloc[-1]
        end_lon = group['longitude'].iloc[-1]
        dist_to_end = haversine_distance(group['latitude'], group['longitude'], end_lat, end_lon)
        df.loc[group.index, 'distance_to_stop'] = dist_to_end
    else:
        df.loc[group.index, 'distance_to_stop'] = 0.0

df = df.dropna(subset=['distance_to_stop']).copy()
print(f"distance_to_stop berhasil dihitung. Ukuran data: {df.shape}")
print(f"Statistik distance_to_stop (km):\n{df['distance_to_stop'].describe()}")

df['speed_clipped'] = df['speed'].clip(lower=1.5)
df['estimated_eta'] = df['distance_to_stop'] / 50 * 60
df['actual_eta'] = df['distance_to_stop'] / df['speed_clipped'] * 60
df['delay_minutes'] = df['actual_eta'] - df['estimated_eta']
print("delay_minutes baru selesai dihitung.")
print(f"Statistik delay_minutes (baru):\n{df['delay_minutes'].describe()}")

le = LabelEncoder()
df['corridor'] = le.fit_transform(df['corridor'].astype(str))

features = ['hour', 'day_of_week', 'corridor', 'distance_to_stop']
X = df[features]
y = df['delay_minutes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

rf_reg_v2 = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg_v2.fit(X_train, y_train)

y_pred = rf_reg_v2.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² Score : {r2:.4f}")
print(f"MAE      : {mae:.2f} menit")
print(f"RMSE     : {rmse:.2f} menit")

for feat, imp in zip(features, rf_reg_v2.feature_importances_):
    print(f"{feat}: {imp:.4f}")

In [ ]:
df['delay_minutes'].quantile([0.95, 0.99, 0.999])

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv('../data/delay_predictor_dataset.csv', low_memory=False)
print(f"Data asli berhasil dimuat. Total baris: {df_raw.shape[0]}")

datetime_col = 'gps_datetime' if 'gps_datetime' in df_raw.columns else 'time'
df_raw[datetime_col] = pd.to_datetime(df_raw[datetime_col])

print("Mengurutkan data secara kronologis...")
df_raw = df_raw.sort_values(by=['trip_id', datetime_col]).copy()

trip_audit = df_raw.groupby('trip_id').agg(
    waktu_mulai=(datetime_col, 'min'),
    waktu_selesai=(datetime_col, 'max'),
    dtd_awal=('dtd', 'first'),   
    dtd_akhir=('dtd', 'last'),   
    dtd_maksimal=('dtd', 'max'), 
    total_titik_gps=('dtd', 'count')
).reset_index()

terpotong_awal = trip_audit[(trip_audit['dtd_maksimal'] - trip_audit['dtd_awal']) > 2.0]
terpotong_akhir = trip_audit[trip_audit['dtd_akhir'] > 1.0]

total_trip = len(trip_audit)
print(f"Total unik `trip_id` di data asli: {total_trip} rute perjalanan")
print(f"Trip terpotong di AWAL (Jam 14:00): {len(terpotong_awal)} trip ({len(terpotong_awal)/total_trip*100:.2f}%)")
print(f"Trip terpotong di AKHIR (Jam 18:00): {len(terpotong_akhir)} trip ({len(terpotong_akhir)/total_trip*100:.2f}%)")

print(terpotong_akhir[['trip_id', 'waktu_mulai', 'waktu_selesai', 'dtd_awal', 'dtd_akhir']].head())

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv('../data/delay_predictor_dataset.csv', low_memory=False)
print(f"Data berhasil dimuat. Total baris asli: {df_raw.shape[0]}")

datetime_col = 'gps_datetime' if 'gps_datetime' in df_raw.columns else 'time'
df_raw[datetime_col] = pd.to_datetime(df_raw[datetime_col])

trip_audit = df_raw.groupby('trip_id').agg(
    waktu_mulai=(datetime_col, 'min'),
    waktu_selesai=(datetime_col, 'max'),
    total_titik_gps=(datetime_col, 'count')
).reset_index()

print("\n=== 3. RUNNING VALIDATED TIME AUDIT ===")
print("Rentang waktu keseluruhan dataset:")
print(df_raw[datetime_col].min(), "sampai", df_raw[datetime_col].max())

print("\nDistribusi waktu_mulai trip (Apakah menumpuk di sekitar 14:00?):")
print(trip_audit['waktu_mulai'].describe())

print("\nDistribusi waktu_selesai trip (Apakah menumpuk di sekitar 18:00?):")
print(trip_audit['waktu_selesai'].describe())

trip_audit['durasi_menit'] = (trip_audit['waktu_selesai'] - trip_audit['waktu_mulai']).dt.total_seconds() / 60
print("\nDistribusi durasi trip yang terekam (Menit):")
print(trip_audit['durasi_menit'].describe())

ambang_pendek = 20
trip_terpotong = trip_audit[trip_audit['durasi_menit'] < ambang_pendek]
pct_terpotong = (len(trip_terpotong) / len(trip_audit)) * 100

print(f"Total unik `trip_id`: {len(trip_audit)} trip")
print(f"Trip dengan durasi < {ambang_pendek} menit (Terpotong) : {len(trip_terpotong)} trip ({pct_terpotong:.2f}%)")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, classification_report

df = pd.read_csv('../data/delay_predictor_dataset.csv', low_memory=False)
df = df[df['corridor'].astype(str) != '0'].copy()
datetime_col = 'gps_datetime' if 'gps_datetime' in df.columns else 'time'
df[datetime_col] = pd.to_datetime(df[datetime_col])
df = df.sort_values(by=['trip_id', datetime_col]).reset_index(drop=True)
print(f"Ukuran data: {df.shape}")

df['time_diff_sec'] = df.groupby('trip_id')[datetime_col].diff().dt.total_seconds()
K_calibration_minutes = (df['time_diff_sec'] / 60).median()
print(f"K_calibration (median interval antar-checkpoint, menit): {K_calibration_minutes:.4f}")
print("^ Ini PURE time-based, tidak melibatkan dtd/jarak sama sekali — aman dari leakage konseptual")

df['speed_clipped'] = df['speed'].clip(lower=1.5)

GROUP_MIN_SIZE = 30  # minimal observasi per grup supaya median dianggap stabil
global_median_speed = df['speed_clipped'].median()

grp = df.groupby(['hour', 'day_of_week', 'corridor'])['speed_clipped']
grp_median = grp.transform('median')
grp_count = grp.transform('count')

df['expected_speed'] = np.where(grp_count >= GROUP_MIN_SIZE, grp_median, global_median_speed)
print(f"Global median speed (fallback): {global_median_speed:.2f} km/h")
print(f"Jumlah baris pakai fallback (grup < {GROUP_MIN_SIZE} obs): {(grp_count < GROUP_MIN_SIZE).sum()} dari {len(df)}")

print("\n=== 5. TARGET PROXY: delay_minutes (TANPA dtd) ===")
speed_deficit_ratio = np.maximum(0, (df['expected_speed'] - df['speed_clipped']) / df['expected_speed'])
df['delay_minutes'] = speed_deficit_ratio * K_calibration_minutes
print(f"Statistik delay_minutes (proxy baru):\n{df['delay_minutes'].describe()}")

print("\n=== 6. STATUS THRESHOLDING ===")
def assign_status(d):
    if d <= 5:
        return "OnTime"
    elif d <= 15:
        return "Late"
    else:
        return "Severe"

df['status'] = df['delay_minutes'].apply(assign_status)
print(df['status'].value_counts(normalize=True))

le = LabelEncoder()
df['corridor_encoded'] = le.fit_transform(df['corridor'].astype(str))

features = ['hour', 'day_of_week', 'corridor_encoded']
X = df[features]
y_reg = df['delay_minutes']
y_clf = df['status']

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_reg_train)
y_pred = rf_reg.predict(X_test)

print(f"R² Score : {r2_score(y_reg_test, y_pred):.4f}")
print(f"MAE      : {mean_absolute_error(y_reg_test, y_pred):.2f} menit")
print(f"RMSE     : {np.sqrt(mean_squared_error(y_reg_test, y_pred)):.2f} menit")

for feat, imp in zip(features, rf_reg.feature_importances_):
    print(f"{feat}: {imp:.4f}")

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
rf_clf.fit(X_train, y_clf_train)
y_clf_pred = rf_clf.predict(X_test)
print(classification_report(y_clf_test, y_clf_pred))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, classification_report

df = pd.read_csv('../data/delay_predictor_dataset.csv', low_memory=False)
df = df[df['corridor'].astype(str) != '0'].copy()
datetime_col = 'gps_datetime' if 'gps_datetime' in df.columns else 'time'
df[datetime_col] = pd.to_datetime(df[datetime_col])
df = df.sort_values(by=['trip_id', datetime_col]).reset_index(drop=True)

# Hitung total durasi dalam menit untuk setiap trip_id
trip_durations = df.groupby('trip_id')[datetime_col].agg(lambda x: (x.max() - x.min()).total_seconds() / 60)
K_calibration_minutes = trip_durations.median()
print(f"K_calibration (median durasi total trip, menit): {K_calibration_minutes:.2f} menit")

df['speed_clipped'] = df['speed'].clip(lower=1.5)

GROUP_MIN_SIZE = 30
global_median_speed = df['speed_clipped'].median()
grp = df.groupby(['hour', 'day_of_week', 'corridor'])['speed_clipped']
grp_median = grp.transform('median')
grp_count = grp.transform('count')
df['expected_speed'] = np.where(grp_count >= GROUP_MIN_SIZE, grp_median, global_median_speed)

speed_deficit_ratio = np.maximum(0, (df['expected_speed'] - df['speed_clipped']) / df['expected_speed'])
df['delay_minutes'] = speed_deficit_ratio * K_calibration_minutes
print(df['delay_minutes'].describe())

def assign_status(d):
    if d <= 5: return "OnTime"
    elif d <= 15: return "Late"
    else: return "Severe"

df['status'] = df['delay_minutes'].apply(assign_status)
print("Distribusi Status Baru:")
print(df['status'].value_counts(normalize=True))

df['speed_lag_1'] = df.groupby('trip_id')['speed_clipped'].shift(1)
df['speed_lag_2'] = df.groupby('trip_id')['speed_clipped'].shift(2)
df['speed_rolling_mean_3'] = df.groupby('trip_id')['speed_clipped'].transform(lambda x: x.rolling(3).mean())

df['speed_lag_1'] = df['speed_lag_1'].fillna(df['expected_speed'])
df['speed_lag_2'] = df['speed_lag_2'].fillna(df['expected_speed'])
df['speed_rolling_mean_3'] = df['speed_rolling_mean_3'].fillna(df['expected_speed'])

le = LabelEncoder()
df['corridor_encoded'] = le.fit_transform(df['corridor'].astype(str))

features = ['hour', 'corridor_encoded', 'speed_lag_1', 'speed_lag_2', 'speed_rolling_mean_3']
X = df[features]
y_reg = df['delay_minutes']
y_clf = df['status']

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_reg_train)
y_pred = rf_reg.predict(X_test)

print(f"R² Score : {r2_score(y_reg_test, y_pred):.4f}")
print(f"MAE      : {mean_absolute_error(y_reg_test, y_pred):.2f} menit")
print(f"RMSE     : {np.sqrt(mean_squared_error(y_reg_test, y_pred)):.2f} menit")

for feat, imp in zip(features, rf_reg.feature_importances_):
    print(f"{feat}: {imp:.4f}")

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
rf_clf.fit(X_train, y_clf_train)
y_clf_pred = rf_clf.predict(X_test)
print(classification_report(y_clf_test, y_clf_pred))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, classification_report

df = pd.read_csv('../data/delay_predictor_dataset.csv', low_memory=False)
df = df[df['corridor'].astype(str) != '0'].copy()
datetime_col = 'gps_datetime' if 'gps_datetime' in df.columns else 'time'
df[datetime_col] = pd.to_datetime(df[datetime_col])
df = df.sort_values(by=['trip_id', datetime_col]).reset_index(drop=True)

K_calibration_minutes = 60.0

df['speed_clipped'] = df['speed'].clip(lower=1.5)

GROUP_MIN_SIZE = 30
global_median_speed = df['speed_clipped'].median()
grp = df.groupby(['hour', 'day_of_week', 'corridor'])['speed_clipped']
grp_median = grp.transform('median')
grp_count = grp.transform('count')
df['expected_speed'] = np.where(grp_count >= GROUP_MIN_SIZE, grp_median, global_median_speed)

speed_deficit_ratio = np.maximum(0, (df['expected_speed'] - df['speed_clipped']) / df['expected_speed'])
df['delay_minutes'] = speed_deficit_ratio * K_calibration_minutes

def assign_status(d):
    if d <= 5:
        return "OnTime"
    elif d <= 15:
        return "Late"
    else:
        return "Severe"

df['status'] = df['delay_minutes'].apply(assign_status)

df['speed_lag_1'] = df.groupby('trip_id')['speed_clipped'].shift(1)
df['speed_lag_2'] = df.groupby('trip_id')['speed_clipped'].shift(2)
df['speed_rolling_mean_3'] = df.groupby('trip_id')['speed_clipped'].transform(lambda x: x.shift(1).rolling(3).mean())

df['speed_lag_1'] = df['speed_lag_1'].fillna(df['expected_speed'])
df['speed_lag_2'] = df['speed_lag_2'].fillna(df['expected_speed'])
df['speed_rolling_mean_3'] = df['speed_rolling_mean_3'].fillna(df['expected_speed'])

le = LabelEncoder()
df['corridor_encoded'] = le.fit_transform(df['corridor'].astype(str))

features = ['hour', 'corridor_encoded', 'speed_lag_1', 'speed_lag_2', 'speed_rolling_mean_3']
X = df[features]
y_reg = df['delay_minutes']
y_clf = df['status']

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_reg_train)
y_pred = rf_reg.predict(X_test)

print(f"R² Score Realistis : {r2_score(y_reg_test, y_pred):.4f}")
print(f"MAE                : {mean_absolute_error(y_reg_test, y_pred):.2f} menit")
print(f"RMSE               : {np.sqrt(mean_squared_error(y_reg_test, y_pred)):.2f} menit")

for feat, imp in zip(features, rf_reg.feature_importances_):
    print(f"{feat}: {imp:.4f}")

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
rf_clf.fit(X_train, y_clf_train)
y_clf_pred = rf_clf.predict(X_test)
print(classification_report(y_clf_test, y_clf_pred))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, classification_report

df = pd.read_csv('../data/delay_predictor_dataset.csv', low_memory=False)
df = df[df['corridor'].astype(str) != '0'].copy()
datetime_col = 'gps_datetime' if 'gps_datetime' in df.columns else 'time'
df[datetime_col] = pd.to_datetime(df[datetime_col])
df = df.sort_values(by=['trip_id', datetime_col]).reset_index(drop=True)

K_calibration_minutes = 60.0

df['speed_clipped'] = df['speed'].clip(lower=1.5)
GROUP_MIN_SIZE = 30
global_median_speed = df['speed_clipped'].median()
grp = df.groupby(['hour', 'day_of_week', 'corridor'])['speed_clipped']
grp_median = grp.transform('median')
grp_count = grp.transform('count')
df['expected_speed'] = np.where(grp_count >= GROUP_MIN_SIZE, grp_median, global_median_speed)

raw_deficit = np.maximum(0, (df['expected_speed'] - df['speed_clipped']) / df['expected_speed'])
df['raw_delay'] = raw_deficit * K_calibration_minutes

df['delay_minutes'] = df.groupby('trip_id')['raw_delay'].transform(lambda x: x.rolling(window=10, min_periods=1).mean())
print(df['delay_minutes'].describe())

def assign_status(d):
    if d <= 5:
        return "OnTime"
    elif d <= 15:
        return "Late"
    else:
        return "Severe"

df['status'] = df['delay_minutes'].apply(assign_status)
print(df['status'].value_counts(normalize=True))

df['speed_lag_1'] = df.groupby('trip_id')['speed_clipped'].shift(1)
df['speed_lag_2'] = df.groupby('trip_id')['speed_clipped'].shift(2)
df['speed_rolling_mean_3'] = df.groupby('trip_id')['speed_clipped'].transform(lambda x: x.shift(1).rolling(3).mean())

df['speed_lag_1'] = df['speed_lag_1'].fillna(df['expected_speed'])
df['speed_lag_2'] = df['speed_lag_2'].fillna(df['expected_speed'])
df['speed_rolling_mean_3'] = df['speed_rolling_mean_3'].fillna(df['expected_speed'])

le = LabelEncoder()
df['corridor_encoded'] = le.fit_transform(df['corridor'].astype(str))

unique_trips = df['trip_id'].unique().tolist()
np.random.seed(42)
np.random.shuffle(unique_trips)

train_size = int(len(unique_trips) * 0.8)
train_trips = unique_trips[:train_size]
test_trips = unique_trips[train_size:]

train_mask = df['trip_id'].isin(train_trips)
test_mask = df['trip_id'].isin(test_trips)

features = ['hour', 'corridor_encoded', 'speed_lag_1', 'speed_lag_2', 'speed_rolling_mean_3']

X_train = df.loc[train_mask, features]
y_reg_train = df.loc[train_mask, 'delay_minutes']
y_clf_train = df.loc[train_mask, 'status']
X_test = df.loc[test_mask, features]
y_reg_test = df.loc[test_mask, 'delay_minutes']
y_clf_test = df.loc[test_mask, 'status']

print(f"Data Train (80% Trip): {X_train.shape}, Data Test (20% Trip): {X_test.shape}")

rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_reg_train)
y_pred = rf_reg.predict(X_test)

print(f"R² Score Hasil Pivot : {r2_score(y_reg_test, y_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_reg_test, y_pred):.2f} menit")
print(f"RMSE: {np.sqrt(mean_squared_error(y_reg_test, y_pred)):.2f} menit")

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
rf_clf.fit(X_train, y_clf_train)
y_clf_pred = rf_clf.predict(X_test)
print(classification_report(y_clf_test, y_clf_pred))


In [ ]:
best_rf_reg = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    min_samples_split=2,
    max_features=None,
    random_state=42,
    n_jobs=-1
)
best_rf_reg.fit(X_train, y_reg_train)
y_reg_pred = best_rf_reg.predict(X_test)
print("R²:", r2_score(y_reg_test, y_reg_pred))
print("MAE:", mean_absolute_error(y_reg_test, y_reg_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_reg_test, y_reg_pred)))
import joblib
joblib.dump(best_rf_reg, "best_rf_reg-v1.pkl") 

In [ ]:
importances = pd.Series(best_rf_reg.feature_importances_, index=features).sort_values(ascending=False)
print(importances)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, classification_report, accuracy_score
import joblib

df = pd.read_csv('../data/delay_predictor_dataset.csv', low_memory=False)
df = df[df['corridor'].astype(str) != '0'].copy()
datetime_col = 'gps_datetime' if 'gps_datetime' in df.columns else 'time'
df[datetime_col] = pd.to_datetime(df[datetime_col])
df = df.sort_values(by=['trip_id', datetime_col]).reset_index(drop=True)

K_calibration_minutes = 60.0

df['speed_clipped'] = df['speed'].clip(lower=1.5)
GROUP_MIN_SIZE = 30
global_median_speed = df['speed_clipped'].median()
grp = df.groupby(['hour', 'day_of_week', 'corridor'])['speed_clipped']
grp_median = grp.transform('median')
grp_count = grp.transform('count')
df['expected_speed'] = np.where(grp_count >= GROUP_MIN_SIZE, grp_median, global_median_speed)

raw_deficit = np.maximum(0, (df['expected_speed'] - df['speed_clipped']) / df['expected_speed'])
df['raw_delay'] = raw_deficit * K_calibration_minutes
df['delay_minutes'] = df.groupby('trip_id')['raw_delay'].transform(lambda x: x.rolling(window=10, min_periods=1).mean())

def assign_status(d):
    if d <= 5: return "OnTime"
    elif d <= 15: return "Late"
    else: return "Severe"
df['status'] = df['delay_minutes'].apply(assign_status)

df['speed_lag_1'] = df.groupby('trip_id')['speed_clipped'].shift(1)
df['speed_lag_2'] = df.groupby('trip_id')['speed_clipped'].shift(2)
df['speed_rolling_mean_3'] = df.groupby('trip_id')['speed_clipped'].transform(lambda x: x.shift(1).rolling(3).mean())

df['speed_lag_1'] = df['speed_lag_1'].fillna(df['expected_speed'])
df['speed_lag_2'] = df['speed_lag_2'].fillna(df['expected_speed'])
df['speed_rolling_mean_3'] = df['speed_rolling_mean_3'].fillna(df['expected_speed'])

le = LabelEncoder()
df['corridor_encoded'] = le.fit_transform(df['corridor'].astype(str))

unique_trips = df['trip_id'].unique().tolist()
np.random.seed(42)
np.random.shuffle(unique_trips)
train_size = int(len(unique_trips) * 0.8)
train_trips = set(unique_trips[:train_size])
test_trips = set(unique_trips[train_size:])

train_mask = df['trip_id'].isin(train_trips)
test_mask = df['trip_id'].isin(test_trips)

features = ['hour', 'speed_lag_1', 'speed_lag_2', 'speed_rolling_mean_3']
X_train, y_reg_train = df.loc[train_mask, features], df.loc[train_mask, 'delay_minutes']
X_test, y_reg_test = df.loc[test_mask, features], df.loc[test_mask, 'delay_minutes']

best_rf_reg = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    min_samples_split=2,
    max_features=None,
    random_state=42,
    n_jobs=-1
)
best_rf_reg.fit(X_train, y_reg_train)

y_reg_pred = best_rf_reg.predict(X_test)
print(f"R²: {r2_score(y_reg_test, y_reg_pred):.5f}")
print(f"MAE: {mean_absolute_error(y_reg_test, y_reg_pred):.5f} menit")
importances = pd.Series(best_rf_reg.feature_importances_, index=features).sort_values(ascending=False)
print(importances)


In [ ]:
import joblib
import json

joblib.dump(best_rf_reg, "best_rf_reg-v1.pkl")
joblib.dump(le, "model1_corridor_encoder_v2.pkl")

expected_speed_lookup = {}
for (hour_val, dow_val, corridor_val), group in df.groupby(['hour', 'day_of_week', 'corridor_encoded']):
    key = f"{hour_val}_{dow_val}_{corridor_val}"
    expected_speed_lookup[key] = float(group['expected_speed'].iloc[0])

model1_config_v2 = {
    "K_calibration_minutes": K_calibration_minutes,
    "delay_thresholds": {"Late": 5, "Severe": 15},
    "features_regressor": ["hour", "corridor_encoded", "speed_lag_1", "speed_lag_2", "speed_rolling_mean_3"],
    "expected_speed_lookup": expected_speed_lookup,
    "global_median_speed": float(global_median_speed)
}

with open("model1_config_v2.json", "w") as f:
    json.dump(model1_config_v2, f, indent=2)

print(f"Jumlah kombinasi unik di expected_speed_lookup: {len(expected_speed_lookup)}")


In [ ]:
import joblib
import pandas as pd
model = joblib.load("../models/model2_crowd_classifier.pkl")
importance = model.feature_importances_

if hasattr(model, 'feature_names_in_'):
    feature_importance = pd.Series(importance, index=model.feature_names_in_)
else:
    feature_importance = pd.Series(importance, index=[f"feature_{i}" for i in range(len(importance))])

print(feature_importance.sort_values(ascending=False))

print(importance)

In [ ]:
FEATURES_NO_CORRIDOR = ['hour', 'speed_lag_1', 'speed_lag_2', 'speed_rolling_mean_3']

X_train_noc = df.loc[train_mask, FEATURES_NO_CORRIDOR]
X_test_noc = df.loc[test_mask, FEATURES_NO_CORRIDOR]

rf_no_corridor = RandomForestRegressor(
    n_estimators=100, max_depth=20, min_samples_leaf=2,
    min_samples_split=2, max_features=None, random_state=42, n_jobs=-1
)
rf_no_corridor.fit(X_train_noc, y_train)

y_pred_noc = rf_no_corridor.predict(X_test_noc)
r2_noc = r2_score(y_test, y_pred_noc)
print(f"R² tanpa corridor: {r2_noc:.4f} (referensi sebelumnya: ~0.126)")

import matplotlib.pyplot as plt

scenarios = ['Dengan corridor', 'Tanpa corridor']
r2_values = [r2, r2_noc]  

plt.figure(figsize=(6,4))
bars = plt.bar(scenarios, r2_values, color=['steelblue', 'lightcoral'])
plt.ylim(0, max(r2_values) * 1.25)
plt.ylabel('R²')
plt.title('Ablation Test Kontribusi Fitur Corridor (Delay Predictor)')
for bar, val in zip(bars, r2_values):
    plt.text(bar.get_x() + bar.get_width() / 2, val + 0.03, f'{val:.3f}', ha='center')
plt.tight_layout()
plt.savefig('fig_ablation_corridor_model1.png', dpi=200)
plt.show()